# Part E — Round 3: Class-weight and decision-threshold optimisation
**Standalone Colab notebook.** Run independently of Rounds 1 and 2. Uses the same processed training and validation CSVs, and does **not** access the test set.

**Method:** 3-fold stratified CV searches class-weight ratios using **Average Precision**; out-of-fold (OOF) training probabilities choose F1/F2 thresholds; fixed choices are then assessed on validation. All exports go to `Part_E_Results/Round_3/`.

**Important:** A different threshold can improve F1/recall without improving Average Precision or ROC-AUC. Validation is for reporting, not selecting a winner repeatedly. OOF thresholds are exploratory and may still be optimistic because hyperparameters were chosen on the same training data.

## 1. Mount Drive and load Part C datasets and prior results

In [1]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight

SEED, FOLDS = 42, 3
if Path('/content').exists():
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
roots = [Path('/content/drive/MyDrive/ML_Group'), Path('/content/drive/MyDrive/ML_Group_Assignment')]
ROOT = next((r for r in roots if (r/'Part_E_Results'/'optimised_metrics.csv').exists()), None)
if ROOT is None: raise FileNotFoundError('Round 1 outputs not found in ML_Group or ML_Group_Assignment')
OUT = ROOT/'Part_E_Results'; R3 = OUT/'Round_3'; R3.mkdir(parents=True,exist_ok=True)
if (ROOT/'train_dataset.csv').exists() and (ROOT/'validation_dataset.csv').exists():
    folder=ROOT
else:
    matches=[p.parent for p in ROOT.rglob('train_dataset.csv') if (p.parent/'validation_dataset.csv').exists()]
    if len(matches)!=1: raise FileNotFoundError(f'Expected one train/validation pair, found {len(matches)}')
    folder=matches[0]
train=pd.read_csv(folder/'train_dataset.csv'); valid=pd.read_csv(folder/'validation_dataset.csv')
assert train.columns.equals(valid.columns) and 'target' in train.columns
X=train.drop(columns='target'); y=train['target'].astype(int)
Xv=valid.drop(columns='target'); yv=valid['target'].astype(int)
assert set(y.unique())=={0,1} and not X.isna().any().any() and not Xv.isna().any().any()
assert X.columns.equals(Xv.columns)
cv=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
balanced=compute_class_weight('balanced',classes=np.array([0,1]),y=y)
balanced_weights={0:float(balanced[0]),1:float(balanced[1])}
balanced_ratio=balanced_weights[1]/balanced_weights[0]
prior=pd.read_csv(OUT/'optimised_metrics.csv').set_index('Model')
round2_path=OUT/'Round_2'/'round2_validation_metrics.csv'
prior2=pd.read_csv(round2_path).set_index('Model') if round2_path.exists() else None
print('Train:',X.shape,'Validation:',Xv.shape,'Class 1 prevalence:',round(y.mean(),4))
print('Original balanced weights:',balanced_weights,'relative positive/negative ratio:',round(balanced_ratio,4))
print('Outputs:',R3)


Mounted at /content/drive
Train: (198221, 66) Validation: (45302, 66) Class 1 prevalence: 0.1552
Original balanced weights: {0: 0.591888227987196, 1: 3.220696714652455} relative positive/negative ratio: 5.4414
Outputs: /content/drive/MyDrive/ML_Group/Part_E_Results/Round_3


## 2. Search class weights (CV Average Precision)
We retain Round 1 winning model structures. Ratios below express the relative positive-class weight with negative-class weight fixed at 1. The original balanced ratio is included; multiplying both class weights by the same constant may not be mathematically identical for every estimator, so we explicitly include the **exact original balanced weights** as a candidate.

In [2]:
# Candidate weighting schemes: exact balanced, plus targeted alternative positive-class weights.
weight_options = [
    ('balanced_original', balanced_weights),
    ('ratio_1', {0:1.0,1:1.0}),
    ('ratio_2', {0:1.0,1:2.0}),
    ('ratio_3_5', {0:1.0,1:3.5}),
    ('ratio_8', {0:1.0,1:8.0}),
]

# Round 1 configurations are deliberately fixed so this experiment isolates class weighting.
models={
 'Logistic Regression': LogisticRegression(C=0.1,penalty='l2',solver='lbfgs',max_iter=1000,random_state=SEED),
 'Histogram Gradient Boosting': HistGradientBoostingClassifier(max_iter=180,learning_rate=0.05,
      max_leaf_nodes=15,l2_regularization=10.0,random_state=SEED),
}
searches={}; cv_summaries=[]
for name,est in models.items():
    print('\nSearching class weights:',name, '|',len(weight_options)*FOLDS,'CV fits')
    t=time.perf_counter()
    gs=GridSearchCV(est, {'class_weight':[weights for _,weights in weight_options]},
        cv=cv,scoring={'ap':'average_precision','roc_auc':'roc_auc','f1':'f1','recall':'recall','precision':'precision'},
        refit='ap',n_jobs=1,verbose=1,error_score='raise',return_train_score=False)
    gs.fit(X,y); searches[name]=gs
    trials=pd.DataFrame(gs.cv_results_)
    trials['weight_scheme']=[next((label for label,w in weight_options if w==p['class_weight']),'other') for p in trials['params']]
    slug='lr' if name=='Logistic Regression' else 'hgb'
    trials.to_csv(R3/f'{slug}_weight_cv_trials.csv',index=False)
    cv_summaries.append({'Model':name,'Best weight scheme':trials.iloc[gs.best_index_]['weight_scheme'],
        'Best class weights':json.dumps(gs.best_params_['class_weight']),
        'CV AP':gs.best_score_,'CV AP std':trials.iloc[gs.best_index_]['std_test_ap'],
        'CV ROC-AUC':trials.iloc[gs.best_index_]['mean_test_roc_auc'],
        'CV F1 @0.50':trials.iloc[gs.best_index_]['mean_test_f1'],
        'Search time (s)':time.perf_counter()-t})
summary=pd.DataFrame(cv_summaries).set_index('Model');display(summary)
summary.reset_index().to_csv(R3/'round3_weight_selection.csv',index=False)



Searching class weights: Logistic Regression | 15 CV fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Searching class weights: Histogram Gradient Boosting | 15 CV fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits


,Best weight scheme,Best class weights,CV AP,CV AP std,CV ROC-AUC,CV F1 @0.50,Search time (s)
Model,,,,,,,
Logistic Regression,ratio_1,"{""0"": 1.0, ""1"": 1.0}",0.288182,0.000387,0.694596,0.032388,35.163920
Histogram Gradient Boosting,ratio_3_5,"{""0"": 1.0, ""1"": 3.5}",0.291741,0.000478,0.697468,0.347985,192.582564


## 3. Select thresholds from out-of-fold **training** probabilities
This uses fresh CV predictions from the selected class-weight configuration, without consulting validation labels. F1 and F2 thresholds are **separate operating points**, not hyperparameters that change probability rankings. OOF scores are not unbiased final performance estimates because the weight configuration was selected using the same training data.

In [3]:
def threshold_from_oof(y_true, probs, beta):
    precision, recall, thresholds = precision_recall_curve(y_true,probs)
    # Precision and recall have one extra terminal point without a threshold.
    p,r=precision[:-1],recall[:-1]
    b2=beta**2
    scores=(1+b2)*p*r/np.maximum(b2*p+r,1e-15)
    # Deterministic tie-break: closest to 0.50.
    maxscore=np.max(scores)
    indices=np.flatnonzero(np.isclose(scores,maxscore,rtol=0,atol=1e-12))
    best=indices[np.argmin(np.abs(thresholds[indices]-0.5))]
    return float(thresholds[best]),float(scores[best])

threshold_rows=[]; oof_probabilities=pd.DataFrame({'actual':y.to_numpy()})
for name,gs in searches.items():
    print('Generating 3-fold OOF probabilities:',name)
    oof=cross_val_predict(clone(gs.best_estimator_),X,y,cv=cv,method='predict_proba',n_jobs=1)[:,1]
    slug='lr' if name=='Logistic Regression' else 'hgb'
    oof_probabilities[f'{slug}_oof_probability']=oof
    t_f1,score_f1=threshold_from_oof(y,oof,1)
    t_f2,score_f2=threshold_from_oof(y,oof,2)
    threshold_rows.append({'Model':name,'OOF F1 threshold':t_f1,'OOF F1 at selected threshold':score_f1,
         'OOF F2 threshold':t_f2,'OOF F2 at selected threshold':score_f2,
         'OOF AP (diagnostic only)':average_precision_score(y,oof)})
thresholds_df=pd.DataFrame(threshold_rows).set_index('Model')
display(thresholds_df.round(5))
thresholds_df.reset_index().to_csv(R3/'round3_oof_thresholds.csv',index=False)
oof_probabilities.to_csv(R3/'round3_oof_train_predictions.csv',index=False)


Generating 3-fold OOF probabilities: Logistic Regression
Generating 3-fold OOF probabilities: Histogram Gradient Boosting


,OOF F1 threshold,OOF F1 at selected threshold,OOF F2 threshold,OOF F2 at selected threshold,OOF AP (diagnostic only)
Model,,,,,
Logistic Regression,0.17623,0.35949,0.09929,0.51956,0.28810
Histogram Gradient Boosting,0.42999,0.36109,0.29722,0.52178,0.29158


## 4. Validation evaluation at fixed thresholds
We report the 0.50 default and both OOF-selected operating points. **Do not choose whichever looks best on validation and then claim it is an unbiased validation result.**

In [4]:
def metrics(y_true, prob, threshold):
    pred=(np.asarray(prob)>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return {'Threshold':threshold,'Accuracy':accuracy_score(y_true,pred),
        'Precision':precision_score(y_true,pred,zero_division=0),
        'Recall':recall_score(y_true,pred,zero_division=0),
        'F1-score':f1_score(y_true,pred,zero_division=0),
        'F2-score':fbeta_score(y_true,pred,beta=2,zero_division=0),
        'ROC-AUC':roc_auc_score(y_true,prob),
        'Average Precision':average_precision_score(y_true,prob),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}

rows=[]; predictions=pd.DataFrame({'actual':yv.to_numpy()})
for name,gs in searches.items():
    prob=gs.best_estimator_.predict_proba(Xv)[:,1]
    slug='lr' if name=='Logistic Regression' else 'hgb'
    predictions[f'{slug}_probability']=prob
    for label,t in [('default_0.50',0.5),('oof_f1',thresholds_df.loc[name,'OOF F1 threshold']),
                    ('oof_f2',thresholds_df.loc[name,'OOF F2 threshold'])]:
        predictions[f'{slug}_prediction_{label}']=(prob>=t).astype(int)
        rows.append({'Model':name,'Operating point':label,**metrics(yv,prob,t)})
results=pd.DataFrame(rows)
results.to_csv(R3/'round3_validation_metrics.csv',index=False)
predictions.to_csv(R3/'round3_validation_predictions.csv',index=False)
display(results[['Model','Operating point','Threshold','Precision','Recall','F1-score','F2-score','ROC-AUC','Average Precision','TP','FP','FN']].round(5))

# Same threshold (0.50) for a fair comparison of weighting alone.
comparison=[]
for name in models:
    now=results[(results.Model==name)&(results['Operating point']=='default_0.50')].iloc[0]
    for source,df in [('Round 1',prior),('Round 2',prior2)]:
        if df is None or name not in df.index:continue
        for met in ['Precision','Recall','F1-score','F2-score','ROC-AUC','Average Precision']:
            comparison.append({'Model':name,'Reference':source,'Metric':met,
                 'Previous @0.50':float(df.loc[name,met]),'Round 3 @0.50':float(now[met]),
                 'Difference':float(now[met])-float(df.loc[name,met])})
compare=pd.DataFrame(comparison)
compare.to_csv(R3/'round3_vs_prior_at_050.csv',index=False)
display(compare.pivot_table(index=['Model','Reference'],columns='Metric',values='Difference').round(5))


,Model,Operating point,Threshold,Precision,Recall,F1-score,F2-score,ROC-AUC,Average Precision,TP,FP,FN
0,Logistic Regression,default_0.50,0.50000,0.42006,0.01904,0.03643,0.02354,0.69468,0.28245,134,185,6903
1,Logistic Regression,oof_f1,0.17623,0.26126,0.57610,0.35949,0.46422,0.69468,0.28245,4054,11463,2983
2,Logistic Regression,oof_f2,0.09929,0.20107,0.86656,0.32641,0.52142,0.69468,0.28245,6098,24229,939
3,Histogram Gradient Boosting,default_0.50,0.50000,0.29313,0.42831,0.34806,0.39214,0.69765,0.28680,3014,7268,4023
4,Histogram Gradient Boosting,oof_f1,0.42999,0.26130,0.59173,0.36251,0.47228,0.69765,0.28680,4164,11772,2873
5,Histogram Gradient Boosting,oof_f2,0.29722,0.21045,0.82905,0.33568,0.52210,0.69765,0.28680,5834,21888,1203


Metric                                 Average Precision  F1-score  F2-score  \
Model                       Reference                                          
Histogram Gradient Boosting Round 1              0.00074  -0.00911  -0.10347   
                            Round 2              0.00083  -0.00910  -0.10309   
Logistic Regression         Round 1              0.00051  -0.32133  -0.46884   
                            Round 2              0.00041  -0.32140  -0.46894   

Metric                                 Precision  ROC-AUC   Recall  
Model                       Reference                               
Histogram Gradient Boosting Round 1      0.04943  0.00002 -0.24002  
                            Round 2      0.04929  0.00047 -0.23888  
Logistic Regression         Round 1      0.17429  0.00017 -0.63820  
                            Round 2      0.17425  0.00006 -0.63834

## 5. Save configurations and concise report
The output preserves the weight scheme selected by CV, the OOF-selected thresholds, and validation results. No final model is selected using validation scores automatically.

In [5]:
config={}
for name,gs in searches.items():
    config[name]={
      'round1_model_structure':gs.best_estimator_.get_params(),
      'selected_class_weight':gs.best_params_['class_weight'],
      'selection_metric':'3-fold training CV average_precision',
      'cv_mean_ap':float(gs.best_score_),
      'oof_f1_threshold':float(thresholds_df.loc[name,'OOF F1 threshold']),
      'oof_f2_threshold':float(thresholds_df.loc[name,'OOF F2 threshold']),
      'validation_not_used_for_selection':True,
      'test_set_accessed':False,
    }
(R3/'round3_model_configurations.json').write_text(json.dumps(config,indent=2,default=str))
lines=['# Part E Round 3 — Class-weight and OOF-threshold experiments','',
       '- CV: 3-fold stratified, shuffled, seed 42. Class-weight selection by Average Precision.',
       '- Model structures fixed at Round 1 winners to isolate weighting.',
       '- Thresholds chosen on out-of-fold training probabilities for F1 and F2.',
       '- Validation is reported at 0.50 and at preselected OOF thresholds.',
       '- Test data never loaded. No automatic winner selected using validation.',
       '- Caution: OOF threshold selection follows CV-based weight selection on the same training data.', '']
for name in models:
    row=summary.loc[name]; th=thresholds_df.loc[name]
    lines += [f'## {name}',f'- Selected weighting: {row["Best weight scheme"]} ({row["Best class weights"]})',
              f'- Mean CV AP: {row["CV AP"]:.6f}',
              f'- OOF F1 threshold: {th["OOF F1 threshold"]:.6f}',
              f'- OOF F2 threshold: {th["OOF F2 threshold"]:.6f}']
    for op in ['default_0.50','oof_f1','oof_f2']:
        r=results[(results.Model==name)&(results['Operating point']==op)].iloc[0]
        lines += [f'- Validation {op}: precision={r["Precision"]:.4f}, recall={r["Recall"]:.4f}, F1={r["F1-score"]:.4f}, F2={r["F2-score"]:.4f}, AP={r["Average Precision"]:.4f}']
    lines.append('')
(R3/'round3_report_summary.md').write_text('\n'.join(lines))
print('Round 3 finished. Saved files:')
for p in sorted(R3.iterdir()): print('-',p.name)


Round 3 finished. Saved files:
- hgb_weight_cv_trials.csv
- lr_weight_cv_trials.csv
- round3_model_configurations.json
- round3_oof_thresholds.csv
- round3_oof_train_predictions.csv
- round3_report_summary.md
- round3_validation_metrics.csv
- round3_validation_predictions.csv
- round3_vs_prior_at_050.csv
- round3_weight_selection.csv
